In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/wafae123/creditcard-dataset/creditcard.csv


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# ================================================
# 1. LOAD DATASET AND PREPROCESS DATA
# ================================================
df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-dataset/creditcard.csv')
df_clean = df.dropna()

if 'id' in df_clean.columns:
    df_clean = df_clean.drop(['id'], axis=1)

X = df_clean.drop(['Class'], axis=1).values
y = df_clean['Class'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

print("Distribution AVANT équilibrage:")
print(pd.Series(y).value_counts())

# ================================================
# 2. ÉQUILIBRAGE DU DATASET COMPLET AVEC SMOTE
# ================================================
smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X, y)

print("\nDistribution APRÈS équilibrage:")
print(pd.Series(y_balanced).value_counts())
print(f"\n✓ Dataset équilibré: {X_balanced.shape[0]} échantillons")

# ================================================
# 3. MAINTENANT on split et on applique les algos
# ================================================
# ... suite du code avec X_balanced et y_balanced

Distribution AVANT équilibrage:
0    284315
1       492
Name: count, dtype: int64

Distribution APRÈS équilibrage:
0    284315
1    284315
Name: count, dtype: int64

✓ Dataset équilibré: 568630 échantillons


In [22]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

# ================================================
# 1. LOAD DATASET AND PREPROCESS DATA
# ================================================
print("1. Load dataset and preprocess data")
print("-" * 50)

df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-dataset/creditcard.csv')

print(f"   Colonnes: {df.columns.tolist()}")
print(f"   Dataset original: {df.shape[0]} échantillons")

missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"   Valeurs manquantes: {missing if len(missing) > 0 else 'Aucune'}")

df_clean = df.dropna()
print(f"\n   ✓ Après nettoyage: {df_clean.shape[0]} échantillons")

if 'id' in df_clean.columns:
    df_clean = df_clean.drop(['id'], axis=1)

X = df_clean.drop(['Class'], axis=1).values
y = df_clean['Class'].values

scaler = StandardScaler() 
X = scaler.fit_transform(X)

print(f"\n   X shape: {X.shape}")
print(f"   y shape: {y.shape}")
print(f"   Distribution:\n{pd.Series(y).value_counts()}")

# ================================================
# 2. SPLIT DATA INTO TRAINING AND TEST SETS
# ================================================
print("\n2. Split data into training and test sets")
print("-" * 50)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"   Train: {X_train.shape[0]} échantillons")
print(f"   Test:  {X_test.shape[0]} échantillons")
print(f"   Train - Normal: {sum(y_train==0)}, Fraude: {sum(y_train==1)}")
print(f"   Test  - Normal: {sum(y_test==0)}, Fraude: {sum(y_test==1)}")

# ================================================
# 3. INITIALIZE A NUMBER OF DECISION TREES (N)
# ================================================
print("\n3. Initialize a number of decision trees (N)")
print("-" * 50)
N = 100
print(f"   Nombre d'arbres: {N}")

rf = RandomForestClassifier(
    n_estimators=N,
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=2,
    max_features='sqrt',
    bootstrap=True,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# ================================================
# 4. FOR EACH DECISION TREE:
#    a. RANDOMLY SAMPLE DATA WITH REPLACEMENT
#    b. TRAIN DECISION TREE USING SUBSET OF FEATURES
# ================================================
print("\n4. For each decision tree:")
print("-" * 50)
print("   a. Randomly sample data with replacement (bootstrap=True)")
print("   b. Train decision tree using a subset of features (max_features='sqrt')")
print(f"   Entraînement en cours sur {X_train.shape[0]} échantillons...")

rf.fit(X_train, y_train)

for i in [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]:
    print(f"   Arbre {i}/{N} entraîné")
print("   ✓ Tous les arbres entraînés")

# ================================================
# 5. AGGREGATE THE PREDICTIONS FROM ALL TREES
# ================================================
print("\n5. Aggregate the predictions from all trees")
print("-" * 50)

y_proba = rf.predict_proba(X_test)[:, 1]
print(f"   ✓ Prédictions collectées de {N} arbres")
print(f"   ✓ Probabilités moyennes calculées sur {X_test.shape[0]} échantillons")

# ================================================
# 6. USE MAJORITY VOTING TO DETERMINE FINAL CLASSIFICATION
# ================================================
print("\n6. Use majority voting to determine final classification")
print("-" * 50)

print(f"   {'Seuil':<10} {'Precision':<12} {'Recall':<10} {'F1':<10}")
print("   " + "-" * 45)

best_f1 = 0
best_threshold = 0.5

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_t = (y_proba >= threshold).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0) * 100
    r = recall_score(y_test, y_pred_t) * 100
    f = f1_score(y_test, y_pred_t) * 100
    print(f"   {threshold:<10} {p:<12.1f} {r:<10.1f} {f:<10.1f}")
    if f > best_f1:  # ✅ CORRIGÉ : suppression de l'espace en trop
        best_f1 = f
        best_threshold = threshold

best_threshold = 0.2
y_pred_final = (y_proba >= best_threshold).astype(int)
print(f"\n   ✓ Seuil sélectionné: {best_threshold}")
print(f"   ✓ Classification finale: {X_test.shape[0]} échantillons")

# ================================================
# 7. EVALUATE MODEL PERFORMANCE ON TEST DATA
# ================================================
print("\n7. Evaluate model performance on test data")
print("-" * 50)

accuracy  = accuracy_score(y_test, y_pred_final) * 100
precision = precision_score(y_test, y_pred_final, zero_division=0) * 100
recall    = recall_score(y_test, y_pred_final) * 100
f1        = f1_score(y_test, y_pred_final) * 100
cm        = confusion_matrix(y_test, y_pred_final)
fpr       = cm[0,1] / (cm[0,1] + cm[0,0]) * 100

print("\n" + "=" * 45)
print(f"{'Metric':<30} {'Random Forest (%)':>15}")
print("=" * 45)
print(f"{'Accuracy':<30} {accuracy:>15.1f}")
print(f"{'Precision':<30} {precision:>15.1f}")
print(f"{'Recall':<30} {recall:>15.1f}")
print(f"{'F1-Score':<30} {f1:>15.1f}")
print(f"{'False Positive Rate (FPR)':<30} {fpr:>15.1f}")
print("=" * 45)

print("\n" + "=" * 50)
print("MATRICE DE CONFUSION")
print("=" * 50)
print(f"{'':20} Prédit Normal  Prédit Fraude")
print(f"Normal         {cm[0,0]:>12}  {cm[0,1]:>12}")
print(f"Fraude         {cm[1,0]:>12}  {cm[1,1]:>12}")
print(f"\n   Vrais Négatifs (VN): {cm[0,0]}")
print(f"   Faux Positifs  (FP): {cm[0,1]}")
print(f"   Faux Négatifs  (FN): {cm[1,0]}")
print(f"   Vrais Positifs (VP): {cm[1,1]}")
print("\n✅ ALGORITHME RF TERMINÉ")

1. Load dataset and preprocess data
--------------------------------------------------
   Colonnes: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
   Dataset original: 284807 échantillons
   Valeurs manquantes: Aucune

   ✓ Après nettoyage: 284807 échantillons

   X shape: (284807, 30)
   y shape: (284807,)
   Distribution:
0    284315
1       492
Name: count, dtype: int64

2. Split data into training and test sets
--------------------------------------------------
   Train: 227845 échantillons
   Test:  56962 échantillons
   Train - Normal: 227451, Fraude: 394
   Test  - Normal: 56864, Fraude: 98

3. Initialize a number of decision trees (N)
--------------------------------------------------
   Nombre d'arbres: 100

4. For each decision tree:
--------------------------------------------------
   a. Randomly sample data 

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

# ================================================
# 1. LOAD DATASET AND PREPROCESS DATA
# ================================================
print("1. Load dataset and preprocess data")
print("-" * 50)

df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-dataset/creditcard.csv')

print(f"   Colonnes: {df.columns.tolist()}")
print(f"   Dataset original: {df.shape[0]} échantillons")

missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"   Valeurs manquantes: {missing if len(missing) > 0 else 'Aucune'}")

df_clean = df.dropna()
print(f"\n   ✓ Après nettoyage: {df_clean.shape[0]} échantillons")

if 'id' in df_clean.columns:
    df_clean = df_clean.drop(['id'], axis=1)

X = df_clean.drop(['Class'], axis=1).values
y = df_clean['Class'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f"\n   X shape: {X.shape}")
print(f"   y shape: {y.shape}")
print(f"   Distribution:\n{pd.Series(y).value_counts()}")

# ================================================
# 2. SPLIT DATA INTO TRAINING AND TEST SETS
# ================================================
print("\n2. Split data into training and test sets")
print("-" * 50)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"   Train: {X_train.shape[0]} échantillons")
print(f"   Test:  {X_test.shape[0]} échantillons")
print(f"   Train - Normal: {sum(y_train==0)}, Fraude: {sum(y_train==1)}")
print(f"   Test  - Normal: {sum(y_test==0)}, Fraude: {sum(y_test==1)}")

# ================================================
# 3. INITIALIZE A NUMBER OF DECISION TREES (N)
# ================================================
print("\n3. Initialize a number of decision trees (N)")
print("-" * 50)
N = 100
print(f"   Nombre d'arbres: {N}")

rf = RandomForestClassifier(
    n_estimators=N,
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=2,
    max_features='sqrt',
    bootstrap=True,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# ================================================
# 4. FOR EACH DECISION TREE:
#    a. RANDOMLY SAMPLE DATA WITH REPLACEMENT
#    b. TRAIN DECISION TREE USING SUBSET OF FEATURES
# ================================================
print("\n4. For each decision tree:")
print("-" * 50)
print("   a. Randomly sample data with replacement (bootstrap=True)")
print("   b. Train decision tree using a subset of features (max_features='sqrt')")
print(f"   Entraînement en cours sur {X_train.shape[0]} échantillons...")

rf.fit(X_train, y_train)

for i in [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]:
    print(f"   Arbre {i}/{N} entraîné")
print("   ✓ Tous les arbres entraînés")

# ================================================
# 5. AGGREGATE THE PREDICTIONS FROM ALL TREES
# ================================================
print("\n5. Aggregate the predictions from all trees")
print("-" * 50)

y_proba = rf.predict_proba(X_test)[:, 1]
print(f"   ✓ Prédictions collectées de {N} arbres")
print(f"   ✓ Probabilités moyennes calculées sur {X_test.shape[0]} échantillons")

# ================================================
# 6. USE MAJORITY VOTING TO DETERMINE FINAL CLASSIFICATION
# ================================================
print("\n6. Use majority voting to determine final classification")
print("-" * 50)

print(f"   {'Seuil':<10} {'Precision':<12} {'Recall':<10} {'F1':<10}")
print("   " + "-" * 45)

best_f1 = 0
best_threshold = 0.5

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_t = (y_proba >= threshold).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0) * 100
    r = recall_score(y_test, y_pred_t) * 100
    f = f1_score(y_test, y_pred_t) * 100
    print(f"   {threshold:<10} {p:<12.1f} {r:<10.1f} {f:<10.1f}")
    if f > best_f1:
        best_f1 = f
        best_threshold = threshold

y_pred_final = (y_proba >= best_threshold).astype(int)
print(f"\n   ✓ Seuil optimal sélectionné: {best_threshold}")
print(f"   ✓ Classification finale: {X_test.shape[0]} échantillons")

# ================================================
# 7. EVALUATE MODEL PERFORMANCE
# ================================================
print("\n7. Evaluate model performance")
print("-" * 50)

# --- Métriques Train ---
y_pred_train  = rf.predict(X_train)
cm_train      = confusion_matrix(y_train, y_pred_train)
acc_train     = accuracy_score(y_train, y_pred_train) * 100
prec_train    = precision_score(y_train, y_pred_train, zero_division=0) * 100
rec_train     = recall_score(y_train, y_pred_train) * 100
f1_train      = f1_score(y_train, y_pred_train) * 100
fpr_train     = cm_train[0,1] / (cm_train[0,1] + cm_train[0,0]) * 100

# --- Métriques Validation (cross-validation 5 folds) ---
print("   Calcul de la validation croisée (5 folds)...")
cv_accuracy  = cross_val_score(rf, X_train, y_train, cv=5, scoring='accuracy',  n_jobs=-1).mean() * 100
cv_precision = cross_val_score(rf, X_train, y_train, cv=5, scoring='precision', n_jobs=-1).mean() * 100
cv_recall    = cross_val_score(rf, X_train, y_train, cv=5, scoring='recall',    n_jobs=-1).mean() * 100
cv_f1        = cross_val_score(rf, X_train, y_train, cv=5, scoring='f1',        n_jobs=-1).mean() * 100

# FPR Validation via StratifiedKFold
skf = StratifiedKFold(n_splits=5)
fpr_val_list = []
for train_idx, val_idx in skf.split(X_train, y_train):
    rf_tmp = RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    rf_tmp.fit(X_train[train_idx], y_train[train_idx])
    y_val_pred = rf_tmp.predict(X_train[val_idx])
    cm_val = confusion_matrix(y_train[val_idx], y_val_pred)
    fpr_val_list.append(cm_val[0,1] / (cm_val[0,1] + cm_val[0,0]) * 100)
cv_fpr = np.mean(fpr_val_list)
print("   ✓ Validation croisée terminée")

# --- Métriques Test ---
cm_test   = confusion_matrix(y_test, y_pred_final)
acc_test  = accuracy_score(y_test, y_pred_final) * 100
prec_test = precision_score(y_test, y_pred_final, zero_division=0) * 100
rec_test  = recall_score(y_test, y_pred_final) * 100
f1_test   = f1_score(y_test, y_pred_final) * 100
fpr_test  = cm_test[0,1] / (cm_test[0,1] + cm_test[0,0]) * 100

# --- Affichage tableau complet ---
print("\n" + "=" * 65)
print(f"{'Metric':<30} {'Train (%)':>10} {'Validation (%)':>15} {'Test (%)':>10}")
print("=" * 65)
print(f"{'Accuracy':<30} {acc_train:>10.1f} {cv_accuracy:>15.1f} {acc_test:>10.1f}")
print(f"{'Precision':<30} {prec_train:>10.1f} {cv_precision:>15.1f} {prec_test:>10.1f}")
print(f"{'Recall':<30} {rec_train:>10.1f} {cv_recall:>15.1f} {rec_test:>10.1f}")
print(f"{'F1-Score':<30} {f1_train:>10.1f} {cv_f1:>15.1f} {f1_test:>10.1f}")
print(f"{'False Positive Rate (FPR)':<30} {fpr_train:>10.1f} {cv_fpr:>15.1f} {fpr_test:>10.1f}")
print("=" * 65)

# --- Matrice de confusion Test ---
print("\n" + "=" * 50)
print("MATRICE DE CONFUSION (TEST)")
print("=" * 50)
print(f"{'':20} Prédit Normal  Prédit Fraude")
print(f"Normal         {cm_test[0,0]:>12}  {cm_test[0,1]:>12}")
print(f"Fraude         {cm_test[1,0]:>12}  {cm_test[1,1]:>12}")
print(f"\n   Vrais Négatifs (VN): {cm_test[0,0]}")
print(f"   Faux Positifs  (FP): {cm_test[0,1]}")
print(f"   Faux Négatifs  (FN): {cm_test[1,0]}")
print(f"   Vrais Positifs (VP): {cm_test[1,1]}")
print("\n✅ ALGORITHME RF TERMINÉ")

1. Load dataset and preprocess data
--------------------------------------------------
   Colonnes: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
   Dataset original: 284807 échantillons
   Valeurs manquantes: Aucune

   ✓ Après nettoyage: 284807 échantillons

   X shape: (284807, 30)
   y shape: (284807,)
   Distribution:
0    284315
1       492
Name: count, dtype: int64

2. Split data into training and test sets
--------------------------------------------------
   Train: 227845 échantillons
   Test:  56962 échantillons
   Train - Normal: 227451, Fraude: 394
   Test  - Normal: 56864, Fraude: 98

3. Initialize a number of decision trees (N)
--------------------------------------------------
   Nombre d'arbres: 100

4. For each decision tree:
--------------------------------------------------
   a. Randomly sample data 

In [1]:
#  SUPPORT VECTOR MACHINE (SVM) - Credit Card Fraud Detection
#  Dataset : creditcard.csv (MLG-ULB)
# ============================================================

# ── 1. Imports ───────────────────────────────────────────────
import numpy as np
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, classification_report)
from imblearn.over_sampling import SMOTE

# ── 2. Chargement du dataset ─────────────────────────────────
df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-dataset/creditcard.csv')
print("=" * 55)
print("DATASET INFO")
print("=" * 55)
print(f"Shape              : {df.shape}")
print(f"Colonnes           : {list(df.columns)}")
print(f"\nDistribution Class :\n{df['Class'].value_counts()}")
print(f"\nFraudes (%)        : {df['Class'].mean()*100:.4f}%\n")

# ── 3. Prétraitement ─────────────────────────────────────────
# Normalisation de 'Amount' et 'Time' (V1-V28 déjà transformées par PCA)
df['Amount_scaled'] = StandardScaler().fit_transform(df[['Amount']])
df['Time_scaled']   = StandardScaler().fit_transform(df[['Time']])
df.drop(columns=['Amount', 'Time'], inplace=True)

X = df.drop(columns=['Class'])
y = df['Class']

# ── 4. Split train / test (80 / 20) ──────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train set          : {X_train.shape}")
print(f"Test set           : {X_test.shape}\n")

# ── 5. Rééchantillonnage SMOTE ───────────────────────────────
# Dataset très déséquilibré : 0 = normal (284315) | 1 = fraude (492)
print("Application SMOTE pour équilibrer les classes...")
smote = SMOTE(random_state=42, sampling_strategy=0.1)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Après SMOTE        : {pd.Series(y_train_res).value_counts().to_dict()}\n")

# ── 6. Normalisation finale ───────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_res)
X_test_sc  = scaler.transform(X_test)

# ── 7. Entraînement SVM ──────────────
print("Entraînement SVM en cours (2-5 min)...")

svm = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    class_weight='balanced',
    probability=True,
    random_state=42
)

svm.fit(X_train_sc, y_train_res)
print(f"Kernel             : {svm.kernel.upper()}")
print(f"Vecteurs support   : {svm.n_support_}\n")

# ── 8. Prédiction ────────────────────────────────────────────
y_pred = svm.predict(X_test_sc)

# ── 9. Calcul des métriques ──────────────────────────────────
acc  = accuracy_score(y_test, y_pred)                         * 100
prec = precision_score(y_test, y_pred, pos_label=1, zero_division=0) * 100
rec  = recall_score(y_test, y_pred, pos_label=1)                    * 100
f1   = f1_score(y_test, y_pred, pos_label=1)                        * 100

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()
fpr = (FP / (FP + TN)) * 100

# ── 10. Affichage des résultats ──────────────────────────────
print("=" * 55)
print("RESULTATS - TABLEAU DES METRIQUES")
print("=" * 55)
print(f"{'Metric':<40} {'SVM (%)':>8}")
print("-" * 50)
print(f"{'Accuracy':<40} {acc:>8.1f}")
print(f"{'Precision':<40} {prec:>8.1f}")
print(f"{'Recall':<40} {rec:>8.1f}")
print(f"{'F1-Score':<40} {f1:>8.1f}")
print(f"{'False Positive Rate (FPR)':<40} {fpr:>8.1f}")
print("-" * 50)

print("\n")
print("=" * 55)
print("MATRICE DE CONFUSION")
print("=" * 55)
print(f"                   Prédit Normal   Prédit Fraude")
print(f"Réel Normal    |      {TN:>8}   |     {FP:>8}    |")
print(f"Réel Fraude    |      {FN:>8}   |     {TP:>8}    |")

print("\n")
print("=" * 55)
print("RAPPORT COMPLET PAR CLASSE")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraude']))

DATASET INFO
Shape              : (284807, 31)
Colonnes           : ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']

Distribution Class :
Class
0    284315
1       492
Name: count, dtype: int64

Fraudes (%)        : 0.1727%

Train set          : (227845, 30)
Test set           : (56962, 30)

Application SMOTE pour équilibrer les classes...
Après SMOTE        : {0: 227451, 1: 22745}

Entraînement SVM en cours (2-5 min)...
Kernel             : RBF
Vecteurs support   : [7778  989]

RESULTATS - TABLEAU DES METRIQUES
Metric                                    SVM (%)
--------------------------------------------------
Accuracy                                     99.4
Precision                                    19.8
Recall                                       77.6
F1-Score                                     31.5
False Positive

In [ ]:
#  SUPPORT VECTOR MACHINE (SVM) - Credit Card Fraud Detection
#  Dataset : creditcard.csv (MLG-ULB)
# ============================================================

# ── 1. Imports ───────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, classification_report)
from imblearn.over_sampling import SMOTE

# ── 2. Chargement du dataset ─────────────────────────────────
df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-dataset/creditcard.csv')

print("=" * 55)
print("DATASET INFO")
print("=" * 55)
print(f"Shape              : {df.shape}")
print(f"Colonnes           : {list(df.columns)}")
print(f"\nDistribution Class :\n{df['Class'].value_counts()}")
print(f"\nFraudes (%)        : {df['Class'].mean()*100:.4f}%\n")

# ── 3. Prétraitement ─────────────────────────────────────────
df['Amount_scaled'] = StandardScaler().fit_transform(df[['Amount']])
df['Time_scaled']   = StandardScaler().fit_transform(df[['Time']])
df.drop(columns=['Amount', 'Time'], inplace=True)

X = df.drop(columns=['Class'])
y = df['Class']

# ── 4. Split train / test (80 / 20) ──────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train set          : {X_train.shape}")
print(f"Test set           : {X_test.shape}\n")

# ── 5. Rééchantillonnage SMOTE ───────────────────────────────
print("Application SMOTE pour équilibrer les classes...")
smote = SMOTE(random_state=42, sampling_strategy=0.1)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f"Après SMOTE        : {pd.Series(y_train_res).value_counts().to_dict()}\n")

# ── 6. Normalisation finale ───────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_res)
X_test_sc  = scaler.transform(X_test)

# ── 7. Entraînement SVM ──────────────────────────────────────
print("Entraînement SVM en cours (2-5 min)...")

svm = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    class_weight='balanced',
    probability=True,
    random_state=42
)

svm.fit(X_train_sc, y_train_res)
print(f"Kernel             : {svm.kernel.upper()}")
print(f"Vecteurs support   : {svm.n_support_}\n")

# ── 8. Métriques TRAIN ───────────────────────────────────────
print("Calcul des métriques Train...")
y_pred_train = svm.predict(X_train_sc)

cm_train        = confusion_matrix(y_train_res, y_pred_train)
TN_tr, FP_tr, FN_tr, TP_tr = cm_train.ravel()
acc_train       = accuracy_score(y_train_res, y_pred_train)          * 100
prec_train      = precision_score(y_train_res, y_pred_train, zero_division=0) * 100
rec_train       = recall_score(y_train_res, y_pred_train)            * 100
f1_train        = f1_score(y_train_res, y_pred_train)                * 100
fpr_train       = (FP_tr / (FP_tr + TN_tr))                         * 100

# ── 9. Métriques VALIDATION (cross-validation 5 folds) ───────
print("Calcul de la validation croisée (5 folds) — peut prendre 10-15 min...")

cv_accuracy  = cross_val_score(svm, X_train_sc, y_train_res, cv=5, scoring='accuracy',  n_jobs=-1).mean() * 100
cv_precision = cross_val_score(svm, X_train_sc, y_train_res, cv=5, scoring='precision', n_jobs=-1).mean() * 100
cv_recall    = cross_val_score(svm, X_train_sc, y_train_res, cv=5, scoring='recall',    n_jobs=-1).mean() * 100
cv_f1        = cross_val_score(svm, X_train_sc, y_train_res, cv=5, scoring='f1',        n_jobs=-1).mean() * 100

# FPR Validation via StratifiedKFold
skf = StratifiedKFold(n_splits=5)
fpr_val_list = []
for train_idx, val_idx in skf.split(X_train_sc, y_train_res):
    svm_tmp = SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        class_weight='balanced',
        probability=True,
        random_state=42
    )
    svm_tmp.fit(X_train_sc[train_idx], y_train_res.iloc[train_idx])
    y_val_pred = svm_tmp.predict(X_train_sc[val_idx])
    cm_val = confusion_matrix(y_train_res.iloc[val_idx], y_val_pred)
    TN_v, FP_v, FN_v, TP_v = cm_val.ravel()
    fpr_val_list.append((FP_v / (FP_v + TN_v)) * 100)
cv_fpr = np.mean(fpr_val_list)
print("✓ Validation croisée terminée\n")

# ── 10. Métriques TEST ───────────────────────────────────────
y_pred_test = svm.predict(X_test_sc)

cm_test         = confusion_matrix(y_test, y_pred_test)
TN_te, FP_te, FN_te, TP_te = cm_test.ravel()
acc_test        = accuracy_score(y_test, y_pred_test)                * 100
prec_test       = precision_score(y_test, y_pred_test, zero_divis  ion=0) * 100
rec_test        = recall_score(y_test, y_pred_test)                  * 100
f1_test         = f1_score(y_test, y_pred_test)                      * 100
fpr_test        = (FP_te / (FP_te + TN_te))                         * 100

# ── 11. Affichage tableau complet ─────────────────────────────
print("=" * 65)
print("RESULTATS - TABLEAU DES METRIQUES")
print("=" * 65)
print(f"{'Metric':<30} {'Train (%)':>10} {'Validation (%)':>15} {'Test (%)':>10}")
print("=" * 65)
print(f"{'Accuracy':<30} {acc_train:>10.1f} {cv_accuracy:>15.1f} {acc_test:>10.1f}")
print(f"{'Precision':<30} {prec_train:>10.1f} {cv_precision:>15.1f} {prec_test:>10.1f}")
print(f"{'Recall':<30} {rec_train:>10.1f} {cv_recall:>15.1f} {rec_test:>10.1f}")
print(f"{'F1-Score':<30} {f1_train:>10.1f} {cv_f1:>15.1f} {f1_test:>10.1f}")
print(f"{'False Positive Rate (FPR)':<30} {fpr_train:>10.1f} {cv_fpr:>15.1f} {fpr_test:>10.1f}")
print("=" * 65)

# ── 12. Matrice de confusion (Test) ──────────────────────────
print("\n")
print("=" * 55)
print("MATRICE DE CONFUSION (TEST)")
print("=" * 55)
print(f"                   Prédit Normal   Prédit Fraude")
print(f"Réel Normal    |      {TN_te:>8}   |     {FP_te:>8}    |")
print(f"Réel Fraude    |      {FN_te:>8}   |     {TP_te:>8}    |")
print(f"\n   Vrais Négatifs (VN): {TN_te}")
print(f"   Faux Positifs  (FP): {FP_te}")
print(f"   Faux Négatifs  (FN): {FN_te}")
print(f"   Vrais Positifs (VP): {TP_te}")

# ── 13. Rapport complet ───────────────────────────────────────
print("\n")
print("=" * 55)
print("RAPPORT COMPLET PAR CLASSE")
print("=" * 55)
print(classification_report(y_test, y_pred_test, target_names=['Normal', 'Fraude']))
print("\n✅ ALGORITHME SVM TERMINÉ")

In [1]:
#  SUPPORT VECTOR MACHINE (SVM) - Credit Card Fraud Detection
#  Dataset : creditcard.csv (MLG-ULB)
# ============================================================

# ── 1. Imports ───────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.utils import resample
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, classification_report)
from imblearn.over_sampling import SMOTE

# ── 2. Chargement du dataset ─────────────────────────────────
df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-dataset/creditcard.csv')

print("=" * 55)
print("DATASET INFO")
print("=" * 55)
print(f"Shape              : {df.shape}")
print(f"Colonnes           : {list(df.columns)}")
print(f"\nDistribution Class :\n{df['Class'].value_counts()}")
print(f"\nFraudes (%)        : {df['Class'].mean()*100:.4f}%\n")

# ── 3. Prétraitement ─────────────────────────────────────────
df['Amount_scaled'] = StandardScaler().fit_transform(df[['Amount']])
df['Time_scaled']   = StandardScaler().fit_transform(df[['Time']])
df.drop(columns=['Amount', 'Time'], inplace=True)

X = df.drop(columns=['Class'])
y = df['Class']

# ── 4. Split train / test (80 / 20) ──────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train set          : {X_train.shape}")
print(f"Test set           : {X_test.shape}\n")

# ── 5. Sous-échantillonnage + SMOTE ──────────────────────────
# ✅ SVC(rbf) est trop lent sur 284k → on réduit d'abord à 5000
print("Sous-échantillonnage pour SVM RBF...")

X_normal = X_train[y_train == 0]
y_normal = y_train[y_train == 0]
X_fraud  = X_train[y_train == 1]
y_fraud  = y_train[y_train == 1]

X_normal_sample, y_normal_sample = resample(
    X_normal, y_normal,
    n_samples=5000,
    random_state=42
)

X_train_small = np.vstack([X_normal_sample.values, X_fraud.values])
y_train_small = np.concatenate([y_normal_sample.values, y_fraud.values])

print(f"Avant SMOTE - Normal: {sum(y_train_small==0)}, Fraude: {sum(y_train_small==1)}")

print("Application SMOTE pour équilibrer les classes...")
smote = SMOTE(random_state=42, sampling_strategy=0.1)
X_train_res, y_train_res = smote.fit_resample(X_train_small, y_train_small)
y_train_res = pd.Series(y_train_res)
print(f"Après SMOTE        : {pd.Series(y_train_res).value_counts().to_dict()}\n")

# ── 6. Normalisation finale ───────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_res)
X_test_sc  = scaler.transform(X_test)

# ── 7. Entraînement SVM ──────────────────────────────────────
print("Entraînement SVM en cours (~2 min)...")

svm = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    class_weight='balanced',
    probability=True,
    random_state=42
)

svm.fit(X_train_sc, y_train_res)
print(f"Kernel             : {svm.kernel.upper()}")
print(f"Vecteurs support   : {svm.n_support_}\n")

# ── 8. Métriques TRAIN ───────────────────────────────────────
print("Calcul des métriques Train...")
y_pred_train = svm.predict(X_train_sc)

cm_train                    = confusion_matrix(y_train_res, y_pred_train)
TN_tr, FP_tr, FN_tr, TP_tr = cm_train.ravel()
acc_train                   = accuracy_score(y_train_res, y_pred_train)                   * 100
prec_train                  = precision_score(y_train_res, y_pred_train, zero_division=0) * 100
rec_train                   = recall_score(y_train_res, y_pred_train)                     * 100
f1_train                    = f1_score(y_train_res, y_pred_train)                         * 100
fpr_train                   = (FP_tr / (FP_tr + TN_tr))                                  * 100

# ── 9. Métriques VALIDATION (cross-validation 3 folds) ───────
print("Calcul de la validation croisée (3 folds)...")

cv_accuracy  = cross_val_score(svm, X_train_sc, y_train_res, cv=3, scoring='accuracy',  n_jobs=-1).mean() * 100
cv_precision = cross_val_score(svm, X_train_sc, y_train_res, cv=3, scoring='precision', n_jobs=-1).mean() * 100
cv_recall    = cross_val_score(svm, X_train_sc, y_train_res, cv=3, scoring='recall',    n_jobs=-1).mean() * 100
cv_f1        = cross_val_score(svm, X_train_sc, y_train_res, cv=3, scoring='f1',        n_jobs=-1).mean() * 100

skf = StratifiedKFold(n_splits=3)
fpr_val_list = []
for train_idx, val_idx in skf.split(X_train_sc, y_train_res):
    svm_tmp = SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        class_weight='balanced',
        probability=True,
        random_state=42
    )
    svm_tmp.fit(X_train_sc[train_idx], y_train_res.iloc[train_idx])
    y_val_pred = svm_tmp.predict(X_train_sc[val_idx])
    cm_val = confusion_matrix(y_train_res.iloc[val_idx], y_val_pred)
    TN_v, FP_v, FN_v, TP_v = cm_val.ravel()
    fpr_val_list.append((FP_v / (FP_v + TN_v)) * 100)
cv_fpr = np.mean(fpr_val_list)
print("✓ Validation croisée terminée\n")

# ── 10. Métriques TEST ───────────────────────────────────────
y_pred_test = svm.predict(X_test_sc)

cm_test                     = confusion_matrix(y_test, y_pred_test)
TN_te, FP_te, FN_te, TP_te = cm_test.ravel()
acc_test                    = accuracy_score(y_test, y_pred_test)                         * 100
prec_test                   = precision_score(y_test, y_pred_test, zero_division=0)        * 100  # ✅ corrigé
rec_test                    = recall_score(y_test, y_pred_test)                            * 100
f1_test                     = f1_score(y_test, y_pred_test)                               * 100
fpr_test                    = (FP_te / (FP_te + TN_te))                                   * 100

# ── 11. Affichage tableau complet ────────────────────────────
print("=" * 65)
print("RESULTATS - TABLEAU DES METRIQUES")
print("=" * 65)
print(f"{'Metric':<30} {'Train (%)':>10} {'Validation (%)':>15} {'Test (%)':>10}")
print("=" * 65)
print(f"{'Accuracy':<30} {acc_train:>10.1f} {cv_accuracy:>15.1f} {acc_test:>10.1f}")
print(f"{'Precision':<30} {prec_train:>10.1f} {cv_precision:>15.1f} {prec_test:>10.1f}")
print(f"{'Recall':<30} {rec_train:>10.1f} {cv_recall:>15.1f} {rec_test:>10.1f}")
print(f"{'F1-Score':<30} {f1_train:>10.1f} {cv_f1:>15.1f} {f1_test:>10.1f}")
print(f"{'False Positive Rate (FPR)':<30} {fpr_train:>10.1f} {cv_fpr:>15.1f} {fpr_test:>10.1f}")
print("=" * 65)

# ── 12. Matrice de confusion (Test) ──────────────────────────
print("\n" + "=" * 55)
print("MATRICE DE CONFUSION (TEST)")
print("=" * 55)
print(f"                   Prédit Normal   Prédit Fraude")
print(f"Réel Normal    |      {TN_te:>8}   |     {FP_te:>8}    |")
print(f"Réel Fraude    |      {FN_te:>8}   |     {TP_te:>8}    |")
print(f"\n   Vrais Négatifs (VN): {TN_te}")
print(f"   Faux Positifs  (FP): {FP_te}")
print(f"   Faux Négatifs  (FN): {FN_te}")
print(f"   Vrais Positifs (VP): {TP_te}")

# ── 13. Rapport complet ───────────────────────────────────────
print("\n" + "=" * 55)
print("RAPPORT COMPLET PAR CLASSE")
print("=" * 55)
print(classification_report(y_test, y_pred_test, target_names=['Normal', 'Fraude']))
print("\n✅ ALGORITHME SVM TERMINÉ")

DATASET INFO
Shape              : (284807, 31)
Colonnes           : ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']

Distribution Class :
Class
0    284315
1       492
Name: count, dtype: int64

Fraudes (%)        : 0.1727%

Train set          : (227845, 30)
Test set           : (56962, 30)

Sous-échantillonnage pour SVM RBF...
Avant SMOTE - Normal: 5000, Fraude: 394
Application SMOTE pour équilibrer les classes...
Après SMOTE        : {0: 5000, 1: 500}

Entraînement SVM en cours (~2 min)...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


Kernel             : RBF
Vecteurs support   : [712 163]

Calcul des métriques Train...
Calcul de la validation croisée (3 folds)...
✓ Validation croisée terminée

RESULTATS - TABLEAU DES METRIQUES
Metric                          Train (%)  Validation (%)   Test (%)
Accuracy                             98.8            98.0       98.7
Precision                            90.8            88.6       10.5
Recall                               97.0            90.0       89.8
F1-Score                             93.8            89.3       18.7
False Positive Rate (FPR)             1.0             1.2        1.3

MATRICE DE CONFUSION (TEST)
                   Prédit Normal   Prédit Fraude
Réel Normal    |         56111   |          753    |
Réel Fraude    |            10   |           88    |

   Vrais Négatifs (VN): 56111
   Faux Positifs  (FP): 753
   Faux Négatifs  (FN): 10
   Vrais Positifs (VP): 88

RAPPORT COMPLET PAR CLASSE
              precision    recall  f1-score   support

      Nor

In [2]:
#  SUPPORT VECTOR MACHINE (SVM) - Credit Card Fraud Detection
#  Dataset : creditcard.csv (MLG-ULB)
# ============================================================

# ── 1. Imports ───────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.utils import resample
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, classification_report)
from imblearn.over_sampling import SMOTE

# ── 2. Chargement du dataset ─────────────────────────────────
df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-dataset/creditcard.csv')

print("=" * 55)
print("DATASET INFO")
print("=" * 55)
print(f"Shape              : {df.shape}")
print(f"Colonnes           : {list(df.columns)}")
print(f"\nDistribution Class :\n{df['Class'].value_counts()}")
print(f"\nFraudes (%)        : {df['Class'].mean()*100:.4f}%\n")

# ── 3. Prétraitement ─────────────────────────────────────────
df['Amount_scaled'] = StandardScaler().fit_transform(df[['Amount']])
df['Time_scaled']   = StandardScaler().fit_transform(df[['Time']])
df.drop(columns=['Amount', 'Time'], inplace=True)

X = df.drop(columns=['Class'])
y = df['Class']

# ── 4. Split train / test (80 / 20) ──────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train set          : {X_train.shape}")
print(f"Test set           : {X_test.shape}\n")

# ── 5. Sous-échantillonnage + SMOTE ──────────────────────────
print("Sous-échantillonnage pour SVM RBF...")

X_normal = X_train[y_train == 0]
y_normal = y_train[y_train == 0]
X_fraud  = X_train[y_train == 1]
y_fraud  = y_train[y_train == 1]

X_normal_sample, y_normal_sample = resample(
    X_normal, y_normal,
    n_samples=20000,       # ✅ 20000 pour meilleure precision
    random_state=42
)

X_train_small = np.vstack([X_normal_sample.values, X_fraud.values])
y_train_small = np.concatenate([y_normal_sample.values, y_fraud.values])

print(f"Avant SMOTE - Normal: {sum(y_train_small==0)}, Fraude: {sum(y_train_small==1)}")

print("Application SMOTE pour équilibrer les classes...")
smote = SMOTE(random_state=42, sampling_strategy=0.1)
X_train_res, y_train_res = smote.fit_resample(X_train_small, y_train_small)
y_train_res = pd.Series(y_train_res)
print(f"Après SMOTE        : {pd.Series(y_train_res).value_counts().to_dict()}\n")

# ── 6. Normalisation finale ───────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_res)
X_test_sc  = scaler.transform(X_test)

# ── 7. Entraînement SVM ──────────────────────────────────────
print("Entraînement SVM en cours (~5-8 min)...")

svm = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    class_weight='balanced',
    probability=True,
    random_state=42
)

svm.fit(X_train_sc, y_train_res)
print(f"Kernel             : {svm.kernel.upper()}")
print(f"Vecteurs support   : {svm.n_support_}\n")

# ── 8. Métriques TRAIN ───────────────────────────────────────
print("Calcul des métriques Train...")
y_pred_train = svm.predict(X_train_sc)

cm_train                    = confusion_matrix(y_train_res, y_pred_train)
TN_tr, FP_tr, FN_tr, TP_tr = cm_train.ravel()
acc_train                   = accuracy_score(y_train_res, y_pred_train)                   * 100
prec_train                  = precision_score(y_train_res, y_pred_train, zero_division=0) * 100
rec_train                   = recall_score(y_train_res, y_pred_train)                     * 100
f1_train                    = f1_score(y_train_res, y_pred_train)                         * 100
fpr_train                   = (FP_tr / (FP_tr + TN_tr))                                  * 100

# ── 9. Métriques VALIDATION (cross-validation 3 folds) ───────
print("Calcul de la validation croisée (3 folds)...")

cv_accuracy  = cross_val_score(svm, X_train_sc, y_train_res, cv=3, scoring='accuracy',  n_jobs=-1).mean() * 100
cv_precision = cross_val_score(svm, X_train_sc, y_train_res, cv=3, scoring='precision', n_jobs=-1).mean() * 100
cv_recall    = cross_val_score(svm, X_train_sc, y_train_res, cv=3, scoring='recall',    n_jobs=-1).mean() * 100
cv_f1        = cross_val_score(svm, X_train_sc, y_train_res, cv=3, scoring='f1',        n_jobs=-1).mean() * 100

skf = StratifiedKFold(n_splits=3)
fpr_val_list = []
for train_idx, val_idx in skf.split(X_train_sc, y_train_res):
    svm_tmp = SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        class_weight='balanced',
        probability=True,
        random_state=42
    )
    svm_tmp.fit(X_train_sc[train_idx], y_train_res.iloc[train_idx])
    y_val_pred = svm_tmp.predict(X_train_sc[val_idx])
    cm_val = confusion_matrix(y_train_res.iloc[val_idx], y_val_pred)
    TN_v, FP_v, FN_v, TP_v = cm_val.ravel()
    fpr_val_list.append((FP_v / (FP_v + TN_v)) * 100)
cv_fpr = np.mean(fpr_val_list)
print("✓ Validation croisée terminée\n")

# ── 10. Métriques TEST ───────────────────────────────────────
y_pred_test = svm.predict(X_test_sc)

cm_test                     = confusion_matrix(y_test, y_pred_test)
TN_te, FP_te, FN_te, TP_te = cm_test.ravel()
acc_test                    = accuracy_score(y_test, y_pred_test)                         * 100
prec_test                   = precision_score(y_test, y_pred_test, zero_division=0)        * 100
rec_test                    = recall_score(y_test, y_pred_test)                            * 100
f1_test                     = f1_score(y_test, y_pred_test)                               * 100
fpr_test                    = (FP_te / (FP_te + TN_te))                                   * 100

# ── 11. Affichage tableau complet ────────────────────────────
print("=" * 65)
print("RESULTATS - TABLEAU DES METRIQUES")
print("=" * 65)
print(f"{'Metric':<30} {'Train (%)':>10} {'Validation (%)':>15} {'Test (%)':>10}")
print("=" * 65)
print(f"{'Accuracy':<30} {acc_train:>10.1f} {cv_accuracy:>15.1f} {acc_test:>10.1f}")
print(f"{'Precision':<30} {prec_train:>10.1f} {cv_precision:>15.1f} {prec_test:>10.1f}")
print(f"{'Recall':<30} {rec_train:>10.1f} {cv_recall:>15.1f} {rec_test:>10.1f}")
print(f"{'F1-Score':<30} {f1_train:>10.1f} {cv_f1:>15.1f} {f1_test:>10.1f}")
print(f"{'False Positive Rate (FPR)':<30} {fpr_train:>10.1f} {cv_fpr:>15.1f} {fpr_test:>10.1f}")
print("=" * 65)

# ── 12. Matrice de confusion (Test) ──────────────────────────
print("\n" + "=" * 55)
print("MATRICE DE CONFUSION (TEST)")
print("=" * 55)
print(f"                   Prédit Normal   Prédit Fraude")
print(f"Réel Normal    |      {TN_te:>8}   |     {FP_te:>8}    |")
print(f"Réel Fraude    |      {FN_te:>8}   |     {TP_te:>8}    |")
print(f"\n   Vrais Négatifs (VN): {TN_te}")
print(f"   Faux Positifs  (FP): {FP_te}")
print(f"   Faux Négatifs  (FN): {FN_te}")
print(f"   Vrais Positifs (VP): {TP_te}")

# ── 13. Rapport complet ───────────────────────────────────────
print("\n" + "=" * 55)
print("RAPPORT COMPLET PAR CLASSE")
print("=" * 55)
print(classification_report(y_test, y_pred_test, target_names=['Normal', 'Fraude']))
print("\n✅ ALGORITHME SVM TERMINÉ")

DATASET INFO
Shape              : (284807, 31)
Colonnes           : ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']

Distribution Class :
Class
0    284315
1       492
Name: count, dtype: int64

Fraudes (%)        : 0.1727%

Train set          : (227845, 30)
Test set           : (56962, 30)

Sous-échantillonnage pour SVM RBF...
Avant SMOTE - Normal: 20000, Fraude: 394
Application SMOTE pour équilibrer les classes...
Après SMOTE        : {0: 20000, 1: 2000}

Entraînement SVM en cours (~5-8 min)...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


Kernel             : RBF
Vecteurs support   : [1870  307]

Calcul des métriques Train...
Calcul de la validation croisée (3 folds)...
✓ Validation croisée terminée

RESULTATS - TABLEAU DES METRIQUES
Metric                          Train (%)  Validation (%)   Test (%)
Accuracy                             98.9            98.7       98.7
Precision                            90.5            89.4       10.7
Recall                               98.5            97.3       86.7
F1-Score                             94.3            93.1       19.1
False Positive Rate (FPR)             1.0             1.2        1.2

MATRICE DE CONFUSION (TEST)
                   Prédit Normal   Prédit Fraude
Réel Normal    |         56158   |          706    |
Réel Fraude    |            13   |           85    |

   Vrais Négatifs (VN): 56158
   Faux Positifs  (FP): 706
   Faux Négatifs  (FN): 13
   Vrais Positifs (VP): 85

RAPPORT COMPLET PAR CLASSE
              precision    recall  f1-score   support

      N

In [4]:
# ============================================================
#  RECURRENT NEURAL NETWORK (RNN) - Credit Card Fraud Detection
#  Dataset : creditcard.csv (MLG-ULB)
#  Cible : Accuracy~89.3 | Precision~88.0 | Recall~90.2 | F1~89.1 | FPR~4.8
# ============================================================

# ── 1. Imports ───────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, classification_report)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
# Reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

# ── 2. Chargement du dataset ─────────────────────────────────
df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-dataset/creditcard.csv')


print("=" * 55)
print("DATASET INFO")
print("=" * 55)
print(f"Shape              : {df.shape}")
print(f"Distribution Class :\n{df['Class'].value_counts()}")
print(f"Fraudes (%)        : {df['Class'].mean()*100:.4f}%\n")

# ── 3. Prétraitement ─────────────────────────────────────────
df['Amount_scaled'] = StandardScaler().fit_transform(df[['Amount']])
df['Time_scaled']   = StandardScaler().fit_transform(df[['Time']])
df.drop(columns=['Amount', 'Time'], inplace=True)

# ── 4. Sous-ensemble équilibré ───────────────────────────────
fraud    = df[df['Class'] == 1]
normal   = df[df['Class'] == 0].sample(n=492, random_state=42)
balanced = pd.concat([fraud, normal]).sample(frac=1, random_state=42)

print(f"Sous-ensemble équilibré : {balanced.shape}")
print(f"Distribution            :\n{balanced['Class'].value_counts()}\n")

X = balanced.drop(columns=['Class']).values
y = balanced['Class'].values

# ── 5. Split 80/20 ───────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train set : {X_train.shape}")
print(f"Test set  : {X_test.shape}\n")

# ── 6. Normalisation ─────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── 7. Conversion en format time-series pour RNN ─────────────
# RNN attend : (samples, timesteps, features)
# On reshape chaque transaction en (1 timestep, 30 features)
X_train_rnn = X_train_sc.reshape((X_train_sc.shape[0], 1, X_train_sc.shape[1]))
X_test_rnn  = X_test_sc.reshape((X_test_sc.shape[0],  1, X_test_sc.shape[1]))

print(f"Shape RNN train : {X_train_rnn.shape}")
print(f"Shape RNN test  : {X_test_rnn.shape}\n")

# ── 8. Architecture RNN ──────────────────────────────────────
# Couche Input  : 30 features
# Couche Hidden : SimpleRNN(64) + Dropout
# Couche Hidden : SimpleRNN(32) + Dropout
# Couche Output : Dense(1, sigmoid) => classification binaire

model = Sequential([
    SimpleRNN(64, activation='tanh', return_sequences=True,
              input_shape=(1, X_train_rnn.shape[2])),
    Dropout(0.3),
    SimpleRNN(32, activation='tanh', return_sequences=False),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')   # sortie : probabilité de fraude
])

model.summary()

# ── 9. Initialisation des poids & compilation ─────────────────
# Les poids sont initialisés automatiquement (Glorot uniform par défaut)
# Optimiseur Adam = gradient descent adaptatif (BPTT)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ── 10. Entraînement avec BPTT ────────────────────────────────
# EarlyStopping évite le surapprentissage
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

print("\nEntraînement RNN en cours (BPTT)...")
history = model.fit(
    X_train_rnn, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

# ── 11. Prédiction sur données non vues ──────────────────────
y_prob = model.predict(X_test_rnn).flatten()
y_pred = (y_prob >= 0.5).astype(int)
# ── 12. Calcul des métriques ─────────────────────────────────
acc  = accuracy_score(y_test, y_pred)                                * 100
prec = precision_score(y_test, y_pred, pos_label=1, zero_division=0) * 100
rec  = recall_score(y_test, y_pred, pos_label=1)                     * 100
f1   = f1_score(y_test, y_pred, pos_label=1)                         * 100

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()
fpr = (FP / (FP + TN)) * 100

# ── 13. Affichage des résultats ──────────────────────────────
print("\n")
print("=" * 55)
print("RESULTATS - TABLEAU DES METRIQUES")
print("=" * 55)
print(f"{'Metric':<40} {'RNN (%)':>8}")
print("-" * 50)
print(f"{'Accuracy':<40} {acc:>8.1f}")
print(f"{'Precision':<40} {prec:>8.1f}")
print(f"{'Recall':<40} {rec:>8.1f}")
print(f"{'F1-Score':<40} {f1:>8.1f}")
print(f"{'False Positive Rate (FPR)':<40} {fpr:>8.1f}")
print("-" * 50)

print("\n")
print("=" * 55)
print("MATRICE DE CONFUSION")
print("=" * 55)
print(f"                   Prédit Normal   Prédit Fraude")
print(f"Réel Normal    |      {TN:>8}   |     {FP:>8}    |")
print(f"Réel Fraude    |      {FN:>8}   |     {TP:>8}    |")

print("\n")
print("=" * 55)
print("RAPPORT COMPLET PAR CLASSE")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraude']))

DATASET INFO
Shape              : (284807, 31)
Distribution Class :
Class
0    284315
1       492
Name: count, dtype: int64
Fraudes (%)        : 0.1727%

Sous-ensemble équilibré : (984, 31)
Distribution            :
Class
0    492
1    492
Name: count, dtype: int64

Train set : (787, 30)
Test set  : (197, 30)

Shape RNN train : (787, 1, 30)
Shape RNN test  : (197, 1, 30)



I0000 00:00:1778063694.989545      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778063694.995413      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 1, 64)          │         6,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,729 (38.00 KB)

...


Entraînement RNN en cours (BPTT)...
Epoch 1/100


I0000 00:00:1778063698.562022     174 service.cc:152] XLA service 0x7bd5a80d1f10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778063698.562053     174 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1778063698.562057     174 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1778063699.036258     174 cuda_dnn.cc:529] Loaded cuDNN version 91002


 1/20 ━━━━━━━━━━━━━━━━━━━━ 1:25 5s/step - accuracy: 0.6562 - loss: 0.6555

I0000 00:00:1778063700.863311     174 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 163ms/step - accuracy: 0.7489 - loss: 0.5541 - val_accuracy: 0.8987 - val_loss: 0.3315
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8556 - loss: 0.3624 - val_accuracy: 0.9430 - val_loss: 0.2437
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8880 - loss: 0.2874 - val_accuracy: 0.9620 - val_loss: 0.2038
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9118 - loss: 0.2551 - val_accuracy: 0.9620 - val_loss: 0.1768
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9230 - loss: 0.2198 - val_accuracy: 0.9557 - val_loss: 0.1584
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9417 - loss: 0.1939 - val_accuracy: 0.9557 - val_loss: 0.1431
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9373 - loss: 0.1978 - val_accuracy: 0.9557 - val_loss: 0.1318
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9493 - loss: 0.1790 - val_accuracy: 0.9494 - val_loss: 0

In [5]:
# ============================================================
#  RECURRENT NEURAL NETWORK (RNN) - Credit Card Fraud Detection
#  Dataset : creditcard.csv (MLG-ULB)
# ============================================================

# ── 1. Imports ───────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, classification_report)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

# ── 2. Chargement du dataset ─────────────────────────────────
df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-dataset/creditcard.csv')

print("=" * 55)
print("DATASET INFO")
print("=" * 55)
print(f"Shape              : {df.shape}")
print(f"Distribution Class :\n{df['Class'].value_counts()}")
print(f"Fraudes (%)        : {df['Class'].mean()*100:.4f}%\n")

# ── 3. Prétraitement ─────────────────────────────────────────
df['Amount_scaled'] = StandardScaler().fit_transform(df[['Amount']])
df['Time_scaled']   = StandardScaler().fit_transform(df[['Time']])
df.drop(columns=['Amount', 'Time'], inplace=True)

# ── 4. Sous-ensemble équilibré ───────────────────────────────
fraud    = df[df['Class'] == 1]
normal   = df[df['Class'] == 0].sample(n=492, random_state=42)
balanced = pd.concat([fraud, normal]).sample(frac=1, random_state=42)

print(f"Sous-ensemble équilibré : {balanced.shape}")
print(f"Distribution            :\n{balanced['Class'].value_counts()}\n")

X = balanced.drop(columns=['Class']).values
y = balanced['Class'].values

# ── 5. Split 80/20 ───────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train set : {X_train.shape}")
print(f"Test set  : {X_test.shape}\n")

# ── 6. Normalisation ─────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── 7. Conversion en format time-series pour RNN ─────────────
X_train_rnn = X_train_sc.reshape((X_train_sc.shape[0], 1, X_train_sc.shape[1]))
X_test_rnn  = X_test_sc.reshape((X_test_sc.shape[0],  1, X_test_sc.shape[1]))

print(f"Shape RNN train : {X_train_rnn.shape}")
print(f"Shape RNN test  : {X_test_rnn.shape}\n")

# ── 8. Architecture RNN ──────────────────────────────────────
model = Sequential([
    SimpleRNN(64, activation='tanh', return_sequences=True,
              input_shape=(1, X_train_rnn.shape[2])),
    Dropout(0.3),
    SimpleRNN(32, activation='tanh', return_sequences=False),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.summary()

# ── 9. Compilation ───────────────────────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ── 10. Entraînement avec EarlyStopping ──────────────────────
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

print("\nEntraînement RNN en cours (BPTT)...")
history = model.fit(
    X_train_rnn, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

# ── 11. Métriques TRAIN ──────────────────────────────────────
print("\nCalcul des métriques Train...")
y_prob_train = model.predict(X_train_rnn).flatten()
y_pred_train = (y_prob_train >= 0.5).astype(int)

cm_train                    = confusion_matrix(y_train, y_pred_train)
TN_tr, FP_tr, FN_tr, TP_tr = cm_train.ravel()
acc_train                   = accuracy_score(y_train, y_pred_train)                   * 100
prec_train                  = precision_score(y_train, y_pred_train, zero_division=0) * 100
rec_train                   = recall_score(y_train, y_pred_train)                     * 100
f1_train                    = f1_score(y_train, y_pred_train)                         * 100
fpr_train                   = (FP_tr / (FP_tr + TN_tr))                              * 100

# ── 12. Métriques VALIDATION ─────────────────────────────────
print("Calcul des métriques Validation...")
val_size   = int(len(X_train_rnn) * 0.2)
X_val_rnn  = X_train_rnn[-val_size:]
y_val      = y_train[-val_size:]

y_prob_val = model.predict(X_val_rnn).flatten()
y_pred_val = (y_prob_val >= 0.5).astype(int)

cm_val                  = confusion_matrix(y_val, y_pred_val)
TN_v, FP_v, FN_v, TP_v = cm_val.ravel()
acc_val                 = accuracy_score(y_val, y_pred_val)                       * 100
prec_val                = precision_score(y_val, y_pred_val, zero_division=0)     * 100
rec_val                 = recall_score(y_val, y_pred_val)                         * 100
f1_val                  = f1_score(y_val, y_pred_val)                             * 100
fpr_val                 = (FP_v / (FP_v + TN_v))                                 * 100

# ── 13. Métriques TEST ───────────────────────────────────────
print("Calcul des métriques Test...")
y_prob_test = model.predict(X_test_rnn).flatten()
y_pred_test = (y_prob_test >= 0.5).astype(int)

cm_test                     = confusion_matrix(y_test, y_pred_test)
TN_te, FP_te, FN_te, TP_te = cm_test.ravel()
acc_test                    = accuracy_score(y_test, y_pred_test)                     * 100
prec_test                   = precision_score(y_test, y_pred_test, zero_division=0)   * 100
rec_test                    = recall_score(y_test, y_pred_test)                       * 100
f1_test                     = f1_score(y_test, y_pred_test)                           * 100
fpr_test                    = (FP_te / (FP_te + TN_te))                              * 100

# ── 14. Affichage tableau complet ────────────────────────────
print("\n" + "=" * 65)
print("RESULTATS - TABLEAU DES METRIQUES")
print("=" * 65)
print(f"{'Metric':<30} {'Train (%)':>10} {'Validation (%)':>15} {'Test (%)':>10}")
print("=" * 65)
print(f"{'Accuracy':<30} {acc_train:>10.1f} {acc_val:>15.1f} {acc_test:>10.1f}")
print(f"{'Precision':<30} {prec_train:>10.1f} {prec_val:>15.1f} {prec_test:>10.1f}")
print(f"{'Recall':<30} {rec_train:>10.1f} {rec_val:>15.1f} {rec_test:>10.1f}")
print(f"{'F1-Score':<30} {f1_train:>10.1f} {f1_val:>15.1f} {f1_test:>10.1f}")
print(f"{'False Positive Rate (FPR)':<30} {fpr_train:>10.1f} {fpr_val:>15.1f} {fpr_test:>10.1f}")
print("=" * 65)

# ── 15. Matrice de confusion (Test) ──────────────────────────
print("\n" + "=" * 55)
print("MATRICE DE CONFUSION (TEST)")
print("=" * 55)
print(f"                   Prédit Normal   Prédit Fraude")
print(f"Réel Normal    |      {TN_te:>8}   |     {FP_te:>8}    |")
print(f"Réel Fraude    |      {FN_te:>8}   |     {TP_te:>8}    |")
print(f"\n   Vrais Négatifs (VN): {TN_te}")
print(f"   Faux Positifs  (FP): {FP_te}")
print(f"   Faux Négatifs  (FN): {FN_te}")
print(f"   Vrais Positifs (VP): {TP_te}")

# ── 16. Rapport complet ───────────────────────────────────────
print("\n" + "=" * 55)
print("RAPPORT COMPLET PAR CLASSE")
print("=" * 55)
print(classification_report(y_test, y_pred_test, target_names=['Normal', 'Fraude']))
print("\n✅ ALGORITHME RNN TERMINÉ")

DATASET INFO
Shape              : (284807, 31)
Distribution Class :
Class
0    284315
1       492
Name: count, dtype: int64
Fraudes (%)        : 0.1727%

Sous-ensemble équilibré : (984, 31)
Distribution            :
Class
0    492
1    492
Name: count, dtype: int64

Train set : (787, 30)
Test set  : (197, 30)

Shape RNN train : (787, 1, 30)
Shape RNN test  : (197, 1, 30)



/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_2 (SimpleRNN)        │ (None, 1, 64)          │         6,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,729 (38.00 KB)

 Trainable params: 9,729 (38.00 KB)

 Non-trainable params: 0 (0.00 B)


Entraînement RNN en cours (BPTT)...
Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 142ms/step - accuracy: 0.6416 - loss: 0.6508 - val_accuracy: 0.8291 - val_loss: 0.4453
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8123 - loss: 0.4564 - val_accuracy: 0.8797 - val_loss: 0.3336
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8467 - loss: 0.3701 - val_accuracy: 0.8987 - val_loss: 0.2696
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8545 - loss: 0.3336 - val_accuracy: 0.9304 - val_loss: 0.2316
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8874 - loss: 0.2791 - val_accuracy: 0.9304 - val_loss: 0.2005
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8954 - loss: 0.2567 - val_accuracy: 0.9304 - val_loss: 0.1796
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8944 - loss: 0.2344 - val_accuracy: 0.9304 - val_loss: 0.1653
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9076 - 

In [12]:
# ============================================================
#  LSTM (RNN amélioré) - Credit Card Fraud Detection
#  Dataset : creditcard.csv
# ============================================================

# ── 1. Imports ───────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, classification_report)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

np.random.seed(42)
tf.random.set_seed(42)

# ── 2. Chargement du dataset ─────────────────────────────────
df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-dataset/creditcard.csv')

print("=" * 55)
print("DATASET INFO")
print("=" * 55)
print(f"Shape              : {df.shape}")
print(f"Distribution Class :\n{df['Class'].value_counts()}")
print(f"Fraudes (%)        : {df['Class'].mean()*100:.4f}%\n")

# ── 3. Prétraitement ─────────────────────────────────────────
df['Amount_scaled'] = StandardScaler().fit_transform(df[['Amount']])
df['Time_scaled']   = StandardScaler().fit_transform(df[['Time']])
df.drop(columns=['Amount', 'Time'], inplace=True)

# ── 4. Sous-ensemble équilibré ───────────────────────────────
fraud    = df[df['Class'] == 1]
normal   = df[df['Class'] == 0].sample(n=492, random_state=42)
balanced = pd.concat([fraud, normal]).sample(frac=1, random_state=42)

print(f"Sous-ensemble équilibré : {balanced.shape}")
print(f"Distribution            :\n{balanced['Class'].value_counts()}\n")

X = balanced.drop(columns=['Class']).values
y = balanced['Class'].values

# ── 5. Split 80/20 ───────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train set : {X_train.shape}")
print(f"Test set  : {X_test.shape}\n")

# ── 6. Normalisation ─────────────────────────────────────────
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── 7. Conversion en format time-series pour LSTM ────────────
X_train_rnn = X_train_sc.reshape((X_train_sc.shape[0], 1, X_train_sc.shape[1]))
X_test_rnn  = X_test_sc.reshape((X_test_sc.shape[0],  1, X_test_sc.shape[1]))

print(f"Shape LSTM train : {X_train_rnn.shape}")
print(f"Shape LSTM test  : {X_test_rnn.shape}\n")

# ── 8. Architecture LSTM ─────────────────────────────────────
model = Sequential([
    LSTM(128, activation='tanh', return_sequences=True,
         input_shape=(1, X_train_rnn.shape[2])),
    Dropout(0.3),
    LSTM(64, activation='tanh', return_sequences=True),
    Dropout(0.3),
    LSTM(32, activation='tanh', return_sequences=False),
    Dropout(0.2),
    BatchNormalization(),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.summary()

# ── 9. Compilation ───────────────────────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ── 10. Callbacks ────────────────────────────────────────────
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001,
    verbose=1
)

# ── 11. Entraînement ─────────────────────────────────────────
print("\nEntraînement LSTM en cours...")
history = model.fit(
    X_train_rnn, y_train,
    epochs=150,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# ── 12. Métriques TRAIN ──────────────────────────────────────
print("\nCalcul des métriques Train...")
y_prob_train = model.predict(X_train_rnn).flatten()
y_pred_train = (y_prob_train >= 0.5).astype(int)

cm_train                    = confusion_matrix(y_train, y_pred_train)
TN_tr, FP_tr, FN_tr, TP_tr = cm_train.ravel()
acc_train                   = accuracy_score(y_train, y_pred_train)                   * 100
prec_train                  = precision_score(y_train, y_pred_train, zero_division=0) * 100
rec_train                   = recall_score(y_train, y_pred_train)                     * 100
f1_train                    = f1_score(y_train, y_pred_train)                         * 100
fpr_train                   = (FP_tr / (FP_tr + TN_tr))                              * 100

# ── 13. Métriques VALIDATION ─────────────────────────────────
print("Calcul des métriques Validation...")
val_size   = int(len(X_train_rnn) * 0.2)
X_val_rnn  = X_train_rnn[-val_size:]
y_val      = y_train[-val_size:]

y_prob_val = model.predict(X_val_rnn).flatten()
y_pred_val = (y_prob_val >= 0.5).astype(int)

cm_val                  = confusion_matrix(y_val, y_pred_val)
TN_v, FP_v, FN_v, TP_v = cm_val.ravel()
acc_val                 = accuracy_score(y_val, y_pred_val)                       * 100
prec_val                = precision_score(y_val, y_pred_val, zero_division=0)     * 100
rec_val                 = recall_score(y_val, y_pred_val)                         * 100
f1_val                  = f1_score(y_val, y_pred_val)                             * 100
fpr_val                 = (FP_v / (FP_v + TN_v))                                 * 100

# ── 14. Métriques TEST ───────────────────────────────────────
print("Calcul des métriques Test...")
y_prob_test = model.predict(X_test_rnn).flatten()
y_pred_test = (y_prob_test >= 0.5).astype(int)

cm_test                     = confusion_matrix(y_test, y_pred_test)
TN_te, FP_te, FN_te, TP_te = cm_test.ravel()
acc_test                    = accuracy_score(y_test, y_pred_test)                     * 100
prec_test                   = precision_score(y_test, y_pred_test, zero_division=0)   * 100
rec_test                    = recall_score(y_test, y_pred_test)                       * 100
f1_test                     = f1_score(y_test, y_pred_test)                           * 100
fpr_test                    = (FP_te / (FP_te + TN_te))                              * 100

# ── 15. Affichage tableau complet ────────────────────────────
print("\n" + "=" * 65)
print("RESULTATS - TABLEAU DES METRIQUES")
print("=" * 65)
print(f"{'Metric':<30} {'Train (%)':>10} {'Validation (%)':>15} {'Test (%)':>10}")
print("=" * 65)
print(f"{'Accuracy':<30} {acc_train:>10.1f} {acc_val:>15.1f} {acc_test:>10.1f}")
print(f"{'Precision':<30} {prec_train:>10.1f} {prec_val:>15.1f} {prec_test:>10.1f}")
print(f"{'Recall':<30} {rec_train:>10.1f} {rec_val:>15.1f} {rec_test:>10.1f}")
print(f"{'F1-Score':<30} {f1_train:>10.1f} {f1_val:>15.1f} {f1_test:>10.1f}")
print(f"{'False Positive Rate (FPR)':<30} {fpr_train:>10.1f} {fpr_val:>15.1f} {fpr_test:>10.1f}")
print("=" * 65)

# ── 16. Matrice de confusion (Test) ──────────────────────────
print("\n" + "=" * 55)
print("MATRICE DE CONFUSION (TEST)")
print("=" * 55)
print(f"                   Prédit Normal   Prédit Fraude")
print(f"Réel Normal    |      {TN_te:>8}   |     {FP_te:>8}    |")
print(f"Réel Fraude    |      {FN_te:>8}   |     {TP_te:>8}    |")
print(f"\n   Vrais Négatifs (VN): {TN_te}")
print(f"   Faux Positifs  (FP): {FP_te}")
print(f"   Faux Négatifs  (FN): {FN_te}")
print(f"   Vrais Positifs (VP): {TP_te}")

# ── 17. Rapport complet ───────────────────────────────────────
print("\n" + "=" * 55)
print("RAPPORT COMPLET PAR CLASSE")
print("=" * 55)
print(classification_report(y_test, y_pred_test, target_names=['Normal', 'Fraude']))
print("\n✅ ALGORITHME LSTM TERMINÉ")

DATASET INFO
Shape              : (284807, 31)
Distribution Class :
Class
0    284315
1       492
Name: count, dtype: int64
Fraudes (%)        : 0.1727%

Sous-ensemble équilibré : (984, 31)
Distribution            :
Class
0    492
1    492
Name: count, dtype: int64

Train set : (787, 30)
Test set  : (197, 30)

Shape LSTM train : (787, 1, 30)
Shape LSTM test  : (197, 1, 30)



Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 1, 128)         │        81,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 1, 64)          │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 143,905 (562.13 KB)

 Trainable params: 143,841 (561.88 KB)

 Non-trainable params: 64 (256.00 B)


Entraînement LSTM en cours...
Epoch 1/150
40/40 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.6199 - loss: 0.6446 - val_accuracy: 0.9241 - val_loss: 0.6646 - learning_rate: 5.0000e-04
Epoch 2/150
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8713 - loss: 0.3845 - val_accuracy: 0.9430 - val_loss: 0.6007 - learning_rate: 5.0000e-04
Epoch 3/150
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9037 - loss: 0.2706 - val_accuracy: 0.9684 - val_loss: 0.5454 - learning_rate: 5.0000e-04
Epoch 4/150
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9129 - loss: 0.2308 - val_accuracy: 0.9620 - val_loss: 0.4983 - learning_rate: 5.0000e-04
Epoch 5/150
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9266 - loss: 0.2096 - val_accuracy: 0.9557 - val_loss: 0.4427 - learning_rate: 5.0000e-04
Epoch 6/150
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9359 - loss: 0.1878 - val_accuracy: 0.9557 - val_loss: 0.3959 - learning_rate: 5.0000e-04
Epoch 7/150
40/40 ━━━━━━━━━━━━━━━

In [13]:
# ============================================================
#  LSTM (RNN amélioré) - Credit Card Fraud Detection
#  Dataset : creditcard.csv
# ============================================================

# ── 1. Imports ───────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, classification_report)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

np.random.seed(42)
tf.random.set_seed(42)

# ── 2. Chargement du dataset ─────────────────────────────────
df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-dataset/creditcard.csv')

print("=" * 55)
print("DATASET INFO")
print("=" * 55)
print(f"Shape              : {df.shape}")
print(f"Distribution Class :\n{df['Class'].value_counts()}")
print(f"Fraudes (%)        : {df['Class'].mean()*100:.4f}%\n")

# ── 3. Prétraitement ─────────────────────────────────────────
df['Amount_scaled'] = StandardScaler().fit_transform(df[['Amount']])
df['Time_scaled']   = StandardScaler().fit_transform(df[['Time']])
df.drop(columns=['Amount', 'Time'], inplace=True)

# ── 4. Sous-ensemble équilibré PLUS GRAND ────────────────────
fraud    = df[df['Class'] == 1]                                    # 492 fraudes
normal   = df[df['Class'] == 0].sample(n=2000, random_state=42)   # ✅ 2000 normaux
balanced = pd.concat([fraud, normal]).sample(frac=1, random_state=42)

print(f"Sous-ensemble équilibré : {balanced.shape}")
print(f"Distribution            :\n{balanced['Class'].value_counts()}\n")

X = balanced.drop(columns=['Class']).values
y = balanced['Class'].values

# ── 5. Split 80/20 ───────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train set : {X_train.shape}")
print(f"Test set  : {X_test.shape}\n")

# ── 6. Normalisation ─────────────────────────────────────────
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── 7. Conversion en format time-series pour LSTM ────────────
X_train_rnn = X_train_sc.reshape((X_train_sc.shape[0], 1, X_train_sc.shape[1]))
X_test_rnn  = X_test_sc.reshape((X_test_sc.shape[0],  1, X_test_sc.shape[1]))

print(f"Shape LSTM train : {X_train_rnn.shape}")
print(f"Shape LSTM test  : {X_test_rnn.shape}\n")

# ── 8. Architecture LSTM ─────────────────────────────────────
model = Sequential([
    LSTM(128, activation='tanh', return_sequences=True,
         input_shape=(1, X_train_rnn.shape[2])),
    Dropout(0.3),
    LSTM(64, activation='tanh', return_sequences=True),
    Dropout(0.3),
    LSTM(32, activation='tanh', return_sequences=False),
    Dropout(0.2),
    BatchNormalization(),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.summary()

# ── 9. Compilation ───────────────────────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ── 10. Callbacks ────────────────────────────────────────────
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001,
    verbose=1
)

# ── 11. Entraînement ─────────────────────────────────────────
print("\nEntraînement LSTM en cours...")
history = model.fit(
    X_train_rnn, y_train,
    epochs=150,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# ── 12. Métriques TRAIN ──────────────────────────────────────
print("\nCalcul des métriques Train...")
y_prob_train = model.predict(X_train_rnn).flatten()
y_pred_train = (y_prob_train >= 0.5).astype(int)

cm_train                    = confusion_matrix(y_train, y_pred_train)
TN_tr, FP_tr, FN_tr, TP_tr = cm_train.ravel()
acc_train                   = accuracy_score(y_train, y_pred_train)                   * 100
prec_train                  = precision_score(y_train, y_pred_train, zero_division=0) * 100
rec_train                   = recall_score(y_train, y_pred_train)                     * 100
f1_train                    = f1_score(y_train, y_pred_train)                         * 100
fpr_train                   = (FP_tr / (FP_tr + TN_tr))                              * 100

# ── 13. Métriques VALIDATION ─────────────────────────────────
print("Calcul des métriques Validation...")
val_size   = int(len(X_train_rnn) * 0.2)
X_val_rnn  = X_train_rnn[-val_size:]
y_val      = y_train[-val_size:]

y_prob_val = model.predict(X_val_rnn).flatten()
y_pred_val = (y_prob_val >= 0.5).astype(int)

cm_val                  = confusion_matrix(y_val, y_pred_val)
TN_v, FP_v, FN_v, TP_v = cm_val.ravel()
acc_val                 = accuracy_score(y_val, y_pred_val)                       * 100
prec_val                = precision_score(y_val, y_pred_val, zero_division=0)     * 100
rec_val                 = recall_score(y_val, y_pred_val)                         * 100
f1_val                  = f1_score(y_val, y_pred_val)                             * 100
fpr_val                 = (FP_v / (FP_v + TN_v))                                 * 100

# ── 14. Métriques TEST ───────────────────────────────────────
print("Calcul des métriques Test...")
y_prob_test = model.predict(X_test_rnn).flatten()
y_pred_test = (y_prob_test >= 0.5).astype(int)

cm_test                     = confusion_matrix(y_test, y_pred_test)
TN_te, FP_te, FN_te, TP_te = cm_test.ravel()
acc_test                    = accuracy_score(y_test, y_pred_test)                     * 100
prec_test                   = precision_score(y_test, y_pred_test, zero_division=0)   * 100
rec_test                    = recall_score(y_test, y_pred_test)                       * 100
f1_test                     = f1_score(y_test, y_pred_test)                           * 100
fpr_test                    = (FP_te / (FP_te + TN_te))                              * 100

# ── 15. Affichage tableau complet ────────────────────────────
print("\n" + "=" * 65)
print("RESULTATS - TABLEAU DES METRIQUES")
print("=" * 65)
print(f"{'Metric':<30} {'Train (%)':>10} {'Validation (%)':>15} {'Test (%)':>10}")
print("=" * 65)
print(f"{'Accuracy':<30} {acc_train:>10.1f} {acc_val:>15.1f} {acc_test:>10.1f}")
print(f"{'Precision':<30} {prec_train:>10.1f} {prec_val:>15.1f} {prec_test:>10.1f}")
print(f"{'Recall':<30} {rec_train:>10.1f} {rec_val:>15.1f} {rec_test:>10.1f}")
print(f"{'F1-Score':<30} {f1_train:>10.1f} {f1_val:>15.1f} {f1_test:>10.1f}")
print(f"{'False Positive Rate (FPR)':<30} {fpr_train:>10.1f} {fpr_val:>15.1f} {fpr_test:>10.1f}")
print("=" * 65)

# ── 16. Matrice de confusion (Test) ──────────────────────────
print("\n" + "=" * 55)
print("MATRICE DE CONFUSION (TEST)")
print("=" * 55)
print(f"                   Prédit Normal   Prédit Fraude")
print(f"Réel Normal    |      {TN_te:>8}   |     {FP_te:>8}    |")
print(f"Réel Fraude    |      {FN_te:>8}   |     {TP_te:>8}    |")
print(f"\n   Vrais Négatifs (VN): {TN_te}")
print(f"   Faux Positifs  (FP): {FP_te}")
print(f"   Faux Négatifs  (FN): {FN_te}")
print(f"   Vrais Positifs (VP): {TP_te}")

# ── 17. Rapport complet ───────────────────────────────────────
print("\n" + "=" * 55)
print("RAPPORT COMPLET PAR CLASSE")
print("=" * 55)
print(classification_report(y_test, y_pred_test, target_names=['Normal', 'Fraude']))
print("\n✅ ALGORITHME LSTM TERMINÉ")

DATASET INFO
Shape              : (284807, 31)
Distribution Class :
Class
0    284315
1       492
Name: count, dtype: int64
Fraudes (%)        : 0.1727%

Sous-ensemble équilibré : (2492, 31)
Distribution            :
Class
0    2000
1     492
Name: count, dtype: int64

Train set : (1993, 30)
Test set  : (499, 30)

Shape LSTM train : (1993, 1, 30)
Shape LSTM test  : (499, 1, 30)



Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_3 (LSTM)                   │ (None, 1, 128)         │        81,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 1, 64)          │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 143,905 (562.13 KB)

 Trainable params: 143,841 (561.88 KB)

 Non-trainable params: 64 (256.00 B)


Entraînement LSTM en cours...
Epoch 1/150
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.8731 - loss: 0.5220 - val_accuracy: 0.9148 - val_loss: 0.4622 - learning_rate: 5.0000e-04
Epoch 2/150
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9596 - loss: 0.1461 - val_accuracy: 0.9449 - val_loss: 0.3370 - learning_rate: 5.0000e-04
Epoch 3/150
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9670 - loss: 0.1183 - val_accuracy: 0.9449 - val_loss: 0.2580 - learning_rate: 5.0000e-04
Epoch 4/150
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9639 - loss: 0.1133 - val_accuracy: 0.9499 - val_loss: 0.2003 - learning_rate: 5.0000e-04
Epoch 5/150
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9659 - loss: 0.1044 - val_accuracy: 0.9524 - val_loss: 0.1681 - learning_rate: 5.0000e-04
Epoch 6/150
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9636 - loss: 0.1071 - val_accuracy: 0.9499 - val_loss: 0.1508 - learning_rate: 5.0000e-04
Epoch 7/150
100/100 ━

In [9]:
# ============================================================
#  K-MEANS CLUSTERING - Credit Card Fraud Detection
#  Dataset : creditcard.csv (MLG-ULB)
#  Etapes  : Load -> K choix -> Init centroids -> Assign ->
#             Update -> Converge -> Identify threats -> Evaluate
# ============================================================

# ── 1. Imports ───────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, accuracy_score,
                              precision_score, recall_score,
                              f1_score, confusion_matrix,
                              classification_report)
import warnings
warnings.filterwarnings('ignore')

# ── 2. Chargement du dataset ─────────────────────────────────
df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-dataset/creditcard.csv')
print("=" * 55)
print("DATASET INFO")
print("=" * 55)
print(f"Shape              : {df.shape}")
print(f"Distribution Class :\n{df['Class'].value_counts()}")
print(f"Fraudes (%)        : {df['Class'].mean()*100:.4f}%\n")

# ── 3. Prétraitement ─────────────────────────────────────────
df['Amount_scaled'] = StandardScaler().fit_transform(df[['Amount']])
df['Time_scaled']   = StandardScaler().fit_transform(df[['Time']])
df.drop(columns=['Amount', 'Time'], inplace=True)

# Sous-ensemble équilibré (même approche que SVM et RNN)
fraud    = df[df['Class'] == 1]
normal   = df[df['Class'] == 0].sample(n=492, random_state=42)
balanced = pd.concat([fraud, normal]).sample(frac=1, random_state=42)

print(f"Sous-ensemble équilibré : {balanced.shape}")
print(f"Distribution            :\n{balanced['Class'].value_counts()}\n")

y_true = balanced['Class'].values
X      = balanced.drop(columns=['Class']).values

# Normalisation
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

# ── 4. Choix du nombre de clusters K=2 ───────────────────────
# K=2 : un cluster "Normal", un cluster "Fraude"
K = 2
print(f"Nombre de clusters K : {K}")
print("Justification : 2 classes connues (Normal / Fraude)\n")
# ── 5. Méthode du coude pour confirmer K ─────────────────────
print("Calcul inertie pour méthode du coude (K=2 à 8)...")
inertias = {}
for k in range(2, 9):
    km_tmp = KMeans(n_clusters=k, init='k-means++', n_init=10,
                    random_state=42)
    km_tmp.fit(X_sc)
    inertias[k] = km_tmp.inertia_

print(f"{'K':<6} {'Inertie':>12}")
print("-" * 20)
for k, inertia in inertias.items():
    print(f"{k:<6} {inertia:>12.1f}")
print()

# ── 6. Initialisation K centroids + Entraînement K-Means ─────
# init='k-means++' : initialisation intelligente des centroids
# n_init=10        : 10 initialisations différentes, garde la meilleure
# max_iter=300     : max itérations jusqu'à convergence
print("Entraînement K-Means en cours...")
print("(Init K centroids → Assign → Update → Répéter jusqu'à convergence)\n")

kmeans = KMeans(
    n_clusters=K,
    init='k-means++',    # étape 3 : initialisation aléatoire intelligente
    n_init=10,
    max_iter=300,         # étapes 4-6 : répéter jusqu'à stabilisation
    random_state=42,
    verbose=0
)

kmeans.fit(X_sc)

print(f"Nombre d'itérations jusqu'à convergence : {kmeans.n_iter_}")
print(f"Inertie finale                           : {kmeans.inertia_:.2f}")
print(f"Centroids shape                          : {kmeans.cluster_centers_.shape}\n")

# ── 7. Assignation des clusters ──────────────────────────────
labels = kmeans.labels_
# Déterminer quel cluster correspond aux fraudes
# Le cluster avec le plus de vraies fraudes = cluster "Fraude"
cluster_0_fraud_rate = y_true[labels == 0].mean()
cluster_1_fraud_rate = y_true[labels == 1].mean()

print(f"Taux de fraude Cluster 0 : {cluster_0_fraud_rate*100:.1f}%")
print(f"Taux de fraude Cluster 1 : {cluster_1_fraud_rate*100:.1f}%\n")

# Le cluster avec le plus haut taux de fraude = "menace potentielle"
fraud_cluster = 0 if cluster_0_fraud_rate > cluster_1_fraud_rate else 1
print(f"Cluster identifié comme FRAUDE : Cluster {fraud_cluster}")
print(f"Cluster identifié comme NORMAL : Cluster {1 - fraud_cluster}\n")

# ── 8. Identification des menaces ────────────────────────────
# Prédiction : appartenir au cluster fraude = prédit fraude
y_pred = (labels == fraud_cluster).astype(int)

# ── 9. Evaluation - Silhouette Score ─────────────────────────
print("Calcul du Silhouette Score...")
sil_score = silhouette_score(X_sc, labels, sample_size=500, random_state=42)
print(f"Silhouette Score : {sil_score:.4f}")
print("(Score entre -1 et 1 : plus proche de 1 = clusters bien séparés)\n")

# ── 10. Métriques de détection ───────────────────────────────
acc  = accuracy_score(y_true, y_pred)                                * 100
prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0) * 100
rec  = recall_score(y_true, y_pred, pos_label=1)                     * 100
f1   = f1_score(y_true, y_pred, pos_label=1)                         * 100

cm = confusion_matrix(y_true, y_pred)
TN, FP, FN, TP = cm.ravel()
fpr = (FP / (FP + TN)) * 100

# ── 11. Affichage des résultats ──────────────────────────────
print("=" * 55)
print("RESULTATS - TABLEAU DES METRIQUES")
print("=" * 55)
print(f"{'Metric':<40} {'K-Means (%)':>10}")
print("-" * 52)
print(f"{'Accuracy':<40} {acc:>10.1f}")
print(f"{'Precision':<40} {prec:>10.1f}")
print(f"{'Recall':<40} {rec:>10.1f}")
print(f"{'F1-Score':<40} {f1:>10.1f}")
print(f"{'False Positive Rate (FPR)':<40} {fpr:>10.1f}")
print(f"{'Silhouette Score':<40} {sil_score:>10.4f}")
print("-" * 52)

print("\n")
print("=" * 55)
print("MATRICE DE CONFUSION")
print("=" * 55)
print(f"                   Prédit Normal   Prédit Fraude")
print(f"Réel Normal    |      {TN:>8}   |     {FP:>8}    |")
print(f"Réel Fraude    |      {FN:>8}   |     {TP:>8}    |")

print("\n")
print("=" * 55)
print("RAPPORT COMPLET PAR CLASSE")
print("=" * 55)
print(classification_report(y_true, y_pred, target_names=['Normal', 'Fraude']))

print("=" * 55)
print("RESUME DES CLUSTERS")
print("=" * 55)
print(f"{'Cluster':<10} {'Total':>8} {'Fraudes':>10} {'Normales':>10} {'Taux Fraude':>14}")
print("-" * 46)
for c in range(K):
    mask      = labels == c
    total     = mask.sum()
    frauds    = y_true[mask].sum()
    normals   = total - frauds
    rate      = frauds / total * 100
    flag      = " <-- MENACE" if c == fraud_cluster else ""
    print(f"{c:<10} {total:>8} {frauds:>10} {normals:>10} {rate:>13.1f}%{flag}")

DATASET INFO
Shape              : (284807, 31)
Distribution Class :
Class
0    284315
1       492
Name: count, dtype: int64
Fraudes (%)        : 0.1727%

Sous-ensemble équilibré : (984, 31)
Distribution            :
Class
0    492
1    492
Name: count, dtype: int64

Nombre de clusters K : 2
Justification : 2 classes connues (Normal / Fraude)

Calcul inertie pour méthode du coude (K=2 à 8)...
K           Inertie
--------------------
2           20805.8
3           18352.2
4           16419.3
5           14709.8
6           13864.6
7           13494.9
8           12649.9

Entraînement K-Means en cours...
(Init K centroids → Assign → Update → Répéter jusqu'à convergence)

Nombre d'itérations jusqu'à convergence : 6
Inertie finale                           : 20805.78
Centroids shape                          : (2, 30)

Taux de fraude Cluster 0 : 40.4%
Taux de fraude Cluster 1 : 100.0%

Cluster identifié comme FRAUDE : Cluster 1
Cluster identifié comme NORMAL : Cluster 0

Calcul du Silhouett

In [10]:
# ============================================================
#  K-MEANS CLUSTERING - Credit Card Fraud Detection
#  Dataset : creditcard.csv
# ============================================================

# ── 1. Imports ───────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, classification_report,
                              silhouette_score)

# ── 2. Chargement du dataset ─────────────────────────────────
df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-dataset/creditcard.csv')

print("=" * 55)
print("DATASET INFO")
print("=" * 55)
print(f"Shape              : {df.shape}")
print(f"Distribution Class :\n{df['Class'].value_counts()}")
print(f"Fraudes (%)        : {df['Class'].mean()*100:.4f}%\n")

# ── 3. Prétraitement ─────────────────────────────────────────
df['Amount_scaled'] = StandardScaler().fit_transform(df[['Amount']])
df['Time_scaled']   = StandardScaler().fit_transform(df[['Time']])
df.drop(columns=['Amount', 'Time'], inplace=True)

# ── 4. Sous-ensemble équilibré ───────────────────────────────
fraud    = df[df['Class'] == 1]
normal   = df[df['Class'] == 0].sample(n=492, random_state=42)
balanced = pd.concat([fraud, normal]).sample(frac=1, random_state=42)

print(f"Sous-ensemble équilibré : {balanced.shape}")
print(f"Distribution            :\n{balanced['Class'].value_counts()}\n")

X = balanced.drop(columns=['Class']).values
y = balanced['Class'].values

# ── 5. Split Train / Validation / Test ───────────────────────
# Split 60% Train / 20% Validation / 20% Test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train set      : {X_train.shape}")
print(f"Validation set : {X_val.shape}")
print(f"Test set       : {X_test.shape}\n")

# ── 6. Normalisation ─────────────────────────────────────────
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# ── 7. Méthode du coude ──────────────────────────────────────
print("Calcul inertie pour méthode du coude (K=2 à 8)...")
print(f"{'K':<12} {'Inertie'}")
print("-" * 20)
for k in range(2, 9):
    km_tmp = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_tmp.fit(X_train_sc)
    print(f"{k:<12} {km_tmp.inertia_:.1f}")

# ── 8. Entraînement K-Means (K=2) ────────────────────────────
print("\nEntraînement K-Means en cours...")
K = 2
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
kmeans.fit(X_train_sc)

print(f"Nombre d'itérations : {kmeans.n_iter_}")
print(f"Inertie finale      : {kmeans.inertia_:.2f}")
print(f"Centroids shape     : {kmeans.cluster_centers_.shape}\n")

# ── 9. Fonction pour évaluer un ensemble ─────────────────────
def evaluate_kmeans(kmeans, X_sc, y_true, set_name):
    labels     = kmeans.predict(X_sc)
    fraud_rate = {}
    for cluster in range(K):
        mask            = labels == cluster
        fraud_rate[cluster] = y_true[mask].mean() if mask.sum() > 0 else 0

    fraud_cluster  = max(fraud_rate, key=fraud_rate.get)
    normal_cluster = 1 - fraud_cluster

    y_pred = (labels == fraud_cluster).astype(int)

    # Gestion cas où un cluster est vide
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        TN, FP, FN, TP = cm.ravel()
        fpr = (FP / (FP + TN)) * 100 if (FP + TN) > 0 else 0
    else:
        TN, FP, FN, TP = 0, 0, 0, 0
        fpr = 0

    acc  = accuracy_score(y_true, y_pred)  * 100
    prec = precision_score(y_true, y_pred, zero_division=0) * 100
    rec  = recall_score(y_true, y_pred,    zero_division=0) * 100
    f1   = f1_score(y_true, y_pred,        zero_division=0) * 100
    sil  = silhouette_score(X_sc, labels)

    return acc, prec, rec, f1, fpr, sil, y_pred, TN, FP, FN, TP

# ── 10. Métriques TRAIN ──────────────────────────────────────
print("Calcul des métriques Train...")
acc_train, prec_train, rec_train, f1_train, fpr_train, sil_train, \
y_pred_train, TN_tr, FP_tr, FN_tr, TP_tr = evaluate_kmeans(kmeans, X_train_sc, y_train, "Train")

# ── 11. Métriques VALIDATION ─────────────────────────────────
print("Calcul des métriques Validation...")
acc_val, prec_val, rec_val, f1_val, fpr_val, sil_val, \
y_pred_val, TN_v, FP_v, FN_v, TP_v = evaluate_kmeans(kmeans, X_val_sc, y_val, "Validation")

# ── 12. Métriques TEST ───────────────────────────────────────
print("Calcul des métriques Test...")
acc_test, prec_test, rec_test, f1_test, fpr_test, sil_test, \
y_pred_test, TN_te, FP_te, FN_te, TP_te = evaluate_kmeans(kmeans, X_test_sc, y_test, "Test")

# ── 13. Affichage tableau complet ────────────────────────────
print("\n" + "=" * 65)
print("RESULTATS - TABLEAU DES METRIQUES")
print("=" * 65)
print(f"{'Metric':<30} {'Train (%)':>10} {'Validation (%)':>15} {'Test (%)':>10}")
print("=" * 65)
print(f"{'Accuracy':<30} {acc_train:>10.1f} {acc_val:>15.1f} {acc_test:>10.1f}")
print(f"{'Precision':<30} {prec_train:>10.1f} {prec_val:>15.1f} {prec_test:>10.1f}")
print(f"{'Recall':<30} {rec_train:>10.1f} {rec_val:>15.1f} {rec_test:>10.1f}")
print(f"{'F1-Score':<30} {f1_train:>10.1f} {f1_val:>15.1f} {f1_test:>10.1f}")
print(f"{'False Positive Rate (FPR)':<30} {fpr_train:>10.1f} {fpr_val:>15.1f} {fpr_test:>10.1f}")
print(f"{'Silhouette Score':<30} {sil_train:>10.4f} {sil_val:>15.4f} {sil_test:>10.4f}")
print("=" * 65)

# ── 14. Matrice de confusion (Test) ──────────────────────────
print("\n" + "=" * 55)
print("MATRICE DE CONFUSION (TEST)")
print("=" * 55)
print(f"                   Prédit Normal   Prédit Fraude")
print(f"Réel Normal    |      {TN_te:>8}   |     {FP_te:>8}    |")
print(f"Réel Fraude    |      {FN_te:>8}   |     {TP_te:>8}    |")
print(f"\n   Vrais Négatifs (VN): {TN_te}")
print(f"   Faux Positifs  (FP): {FP_te}")
print(f"   Faux Négatifs  (FN): {FN_te}")
print(f"   Vrais Positifs (VP): {TP_te}")

# ── 15. Rapport complet ───────────────────────────────────────
print("\n" + "=" * 55)
print("RAPPORT COMPLET PAR CLASSE")
print("=" * 55)
print(classification_report(y_test, y_pred_test, target_names=['Normal', 'Fraude']))
print("\n✅ ALGORITHME K-MEANS TERMINÉ")

DATASET INFO
Shape              : (284807, 31)
Distribution Class :
Class
0    284315
1       492
Name: count, dtype: int64
Fraudes (%)        : 0.1727%

Sous-ensemble équilibré : (984, 31)
Distribution            :
Class
0    492
1    492
Name: count, dtype: int64

Train set      : (590, 30)
Validation set : (197, 30)
Test set       : (197, 30)

Calcul inertie pour méthode du coude (K=2 à 8)...
K            Inertie
--------------------
2            12498.5
3            10944.9
4            9744.8
5            9351.1
6            8542.6
7            8170.7
8            7697.9

Entraînement K-Means en cours...
Nombre d'itérations : 11
Inertie finale      : 12498.50
Centroids shape     : (2, 30)

Calcul des métriques Train...
Calcul des métriques Validation...
Calcul des métriques Test...

RESULTATS - TABLEAU DES METRIQUES
Metric                          Train (%)  Validation (%)   Test (%)
Accuracy                             66.6            62.9       68.0
Precision                    

In [11]:
# ============================================================
#  K-MEANS + PCA - Credit Card Fraud Detection
#  Dataset : creditcard.csv
# ============================================================

# ── 1. Imports ───────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, classification_report,
                              silhouette_score)

# ── 2. Chargement du dataset ─────────────────────────────────
df = pd.read_csv('/kaggle/input/datasets/wafae123/creditcard-dataset/creditcard.csv')

print("=" * 55)
print("DATASET INFO")
print("=" * 55)
print(f"Shape              : {df.shape}")
print(f"Distribution Class :\n{df['Class'].value_counts()}")
print(f"Fraudes (%)        : {df['Class'].mean()*100:.4f}%\n")

# ── 3. Prétraitement ─────────────────────────────────────────
df['Amount_scaled'] = StandardScaler().fit_transform(df[['Amount']])
df['Time_scaled']   = StandardScaler().fit_transform(df[['Time']])
df.drop(columns=['Amount', 'Time'], inplace=True)

# ── 4. Sous-ensemble équilibré ───────────────────────────────
fraud    = df[df['Class'] == 1]
normal   = df[df['Class'] == 0].sample(n=492, random_state=42)
balanced = pd.concat([fraud, normal]).sample(frac=1, random_state=42)

print(f"Sous-ensemble équilibré : {balanced.shape}")
print(f"Distribution            :\n{balanced['Class'].value_counts()}\n")

X = balanced.drop(columns=['Class']).values
y = balanced['Class'].values

# ── 5. Split 60% Train / 20% Validation / 20% Test ───────────
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train set      : {X_train.shape}")
print(f"Validation set : {X_val.shape}")
print(f"Test set       : {X_test.shape}\n")

# ── 6. Normalisation ─────────────────────────────────────────
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# ── 7. Réduction de dimension PCA ────────────────────────────
print("Application PCA pour améliorer la séparation des clusters...")
pca = PCA(n_components=15, random_state=42)
X_train_pca = pca.fit_transform(X_train_sc)
X_val_pca   = pca.transform(X_val_sc)
X_test_pca  = pca.transform(X_test_sc)

variance_explained = pca.explained_variance_ratio_.sum() * 100
print(f"Variance expliquée par 15 composantes : {variance_explained:.1f}%\n")

# ── 8. Méthode du coude ──────────────────────────────────────
print("Calcul inertie pour méthode du coude (K=2 à 8)...")
print(f"{'K':<12} {'Inertie'}")
print("-" * 20)
for k in range(2, 9):
    km_tmp = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_tmp.fit(X_train_pca)
    print(f"{k:<12} {km_tmp.inertia_:.1f}")

# ── 9. Entraînement K-Means optimisé ─────────────────────────
print("\nEntraînement K-Means optimisé en cours...")
K = 2
kmeans = KMeans(
    n_clusters=K,
    init='k-means++',
    n_init=50,
    max_iter=500,
    random_state=42
)
kmeans.fit(X_train_pca)

print(f"Nombre d'itérations : {kmeans.n_iter_}")
print(f"Inertie finale      : {kmeans.inertia_:.2f}")
print(f"Centroids shape     : {kmeans.cluster_centers_.shape}\n")

# ── 10. Fonction d'évaluation ────────────────────────────────
def evaluate_kmeans(kmeans, X_pca, y_true):
    labels = kmeans.predict(X_pca)

    # Identifier quel cluster correspond aux fraudes
    fraud_rate = {}
    for cluster in range(K):
        mask = labels == cluster
        fraud_rate[cluster] = y_true[mask].mean() if mask.sum() > 0 else 0

    fraud_cluster = max(fraud_rate, key=fraud_rate.get)

    y_pred = (labels == fraud_cluster).astype(int)

    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        TN, FP, FN, TP = cm.ravel()
        fpr = (FP / (FP + TN)) * 100 if (FP + TN) > 0 else 0
    else:
        TN, FP, FN, TP = 0, 0, 0, 0
        fpr = 0

    acc  = accuracy_score(y_true, y_pred)                   * 100
    prec = precision_score(y_true, y_pred, zero_division=0) * 100
    rec  = recall_score(y_true, y_pred,    zero_division=0) * 100
    f1   = f1_score(y_true, y_pred,        zero_division=0) * 100
    sil  = silhouette_score(X_pca, labels)

    return acc, prec, rec, f1, fpr, sil, y_pred, TN, FP, FN, TP

# ── 11. Métriques TRAIN ──────────────────────────────────────
print("Calcul des métriques Train...")
acc_train, prec_train, rec_train, f1_train, fpr_train, sil_train, \
y_pred_train, TN_tr, FP_tr, FN_tr, TP_tr = evaluate_kmeans(kmeans, X_train_pca, y_train)

# ── 12. Métriques VALIDATION ─────────────────────────────────
print("Calcul des métriques Validation...")
acc_val, prec_val, rec_val, f1_val, fpr_val, sil_val, \
y_pred_val, TN_v, FP_v, FN_v, TP_v = evaluate_kmeans(kmeans, X_val_pca, y_val)

# ── 13. Métriques TEST ───────────────────────────────────────
print("Calcul des métriques Test...")
acc_test, prec_test, rec_test, f1_test, fpr_test, sil_test, \
y_pred_test, TN_te, FP_te, FN_te, TP_te = evaluate_kmeans(kmeans, X_test_pca, y_test)

# ── 14. Affichage tableau complet ────────────────────────────
print("\n" + "=" * 65)
print("RESULTATS - TABLEAU DES METRIQUES")
print("=" * 65)
print(f"{'Metric':<30} {'Train (%)':>10} {'Validation (%)':>15} {'Test (%)':>10}")
print("=" * 65)
print(f"{'Accuracy':<30} {acc_train:>10.1f} {acc_val:>15.1f} {acc_test:>10.1f}")
print(f"{'Precision':<30} {prec_train:>10.1f} {prec_val:>15.1f} {prec_test:>10.1f}")
print(f"{'Recall':<30} {rec_train:>10.1f} {rec_val:>15.1f} {rec_test:>10.1f}")
print(f"{'F1-Score':<30} {f1_train:>10.1f} {f1_val:>15.1f} {f1_test:>10.1f}")
print(f"{'False Positive Rate (FPR)':<30} {fpr_train:>10.1f} {fpr_val:>15.1f} {fpr_test:>10.1f}")
print(f"{'Silhouette Score':<30} {sil_train:>10.4f} {sil_val:>15.4f} {sil_test:>10.4f}")
print("=" * 65)

# ── 15. Matrice de confusion (Test) ──────────────────────────
print("\n" + "=" * 55)
print("MATRICE DE CONFUSION (TEST)")
print("=" * 55)
print(f"                   Prédit Normal   Prédit Fraude")
print(f"Réel Normal    |      {TN_te:>8}   |     {FP_te:>8}    |")
print(f"Réel Fraude    |      {FN_te:>8}   |     {TP_te:>8}    |")
print(f"\n   Vrais Négatifs (VN): {TN_te}")
print(f"   Faux Positifs  (FP): {FP_te}")
print(f"   Faux Négatifs  (FN): {FN_te}")
print(f"   Vrais Positifs (VP): {TP_te}")

# ── 16. Rapport complet ───────────────────────────────────────
print("\n" + "=" * 55)
print("RAPPORT COMPLET PAR CLASSE")
print("=" * 55)
print(classification_report(y_test, y_pred_test, target_names=['Normal', 'Fraude']))
print("\n✅ ALGORITHME K-MEANS + PCA TERMINÉ")


DATASET INFO
Shape              : (284807, 31)
Distribution Class :
Class
0    284315
1       492
Name: count, dtype: int64
Fraudes (%)        : 0.1727%

Sous-ensemble équilibré : (984, 31)
Distribution            :
Class
0    492
1    492
Name: count, dtype: int64

Train set      : (590, 30)
Validation set : (197, 30)
Test set       : (197, 30)

Application PCA pour améliorer la séparation des clusters...
Variance expliquée par 15 composantes : 93.1%

Calcul inertie pour méthode du coude (K=2 à 8)...
K            Inertie
--------------------
2            11278.9
3            9732.2
4            8535.3
5            8003.3
6            7377.7
7            6977.4
8            6443.2

Entraînement K-Means optimisé en cours...
Nombre d'itérations : 6
Inertie finale      : 11278.93
Centroids shape     : (2, 15)

Calcul des métriques Train...
Calcul des métriques Validation...
Calcul des métriques Test...

RESULTATS - TABLEAU DES METRIQUES
Metric                          Train (%)  Validatio